In [1]:
# ─────────────────────────────────────────────────────────────────────
# CELL 1 — Library Imports
# We use:
#   pandas     → DataFrame construction and JSON export
#   numpy      → random distributions
#   random     → weighted choices, sampling
#   Faker      → realistic Indian names, emails, phone numbers
#   datetime   → timestamps
#   pathlib    → folder creation
# ─────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime
from pathlib import Path
from datetime import datetime, timedelta

# pipeline_run_date = datetime.now()
print('Libraries loaded successfully.')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 5, Finished, Available, Finished, False)

Libraries loaded successfully.


In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 — Store Configuration & Random Seeds
#
# Each store gets its own seed so data is deterministic AND different
# from every other store.  Simply change STORE dict + seeds to add a
# new outlet without touching any other cell.
# ─────────────────────────────────────────────────────────────────────

STORE = {
    "store_id":       "S001",
    "store_name":     "Seawoods",
    "store_location": "Seawoods",
    "city":           "Navi Mumbai",
    "state":          "Maharashtra",
    "pincode":        "400706",
}

# Reproducibility seeds — unique per store
RANDOM_SEED  = 42
NUMPY_SEED   = 42

random.seed(RANDOM_SEED)
np.random.seed(NUMPY_SEED)
fake = Faker('en_IN')        # Indian locale for names / contact details
Faker.seed(RANDOM_SEED)

# Operational constants
CUSTOMER_COUNT = 560          # unique customers registered at this store
STAFF_COUNT    = 15           # total staff headcount
ORDER_COUNT    = 680          # orders on the business date (≈ daily volume)
BUSINESS_DATE  = datetime.now()

# BUSINESS_DATE = (datetime.now() - timedelta(days=1)).date()

store_id = STORE['store_id']  # shorthand used in file paths
print(f'Config ready for {STORE["store_name"]} ({store_id}).')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 6, Finished, Available, Finished, False)

Config ready for Seawoods (S001).


In [3]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 — Lookup Tables (Locations & Products)
#
# LOCATIONS   — Navi Mumbai areas where customers live
# DISTANCE_MAP — average km from each area to the store
#               (used for delivery analytics in the Gold layer)
# PRODUCTS    — master SKU list; IDENTICAL across all 5 stores
#               so cross-store product analysis works cleanly
# PRODUCT_WEIGHTS — controls how often each product is picked;
#                   Pizza items weighted highest to hit the
#                   'pizza = majority revenue' business rule
# ─────────────────────────────────────────────────────────────────────

LOCATIONS = [
    'Seawoods', 'Vashi', 'Nerul', 'Kharghar', 'Belapur',
    'Panvel', 'Ulwe', 'Dronagiri', 'Kopar Khairane', 'Turbhe',
]

# Approximate driving distance (km) from each locality to the store
DISTANCE_MAP = {
    'Seawoods':        2.0,
    'Vashi':           5.5,
    'Nerul':           4.0,
    'Kharghar':        8.0,
    'Belapur':         6.5,
    'Panvel':         14.0,
    'Ulwe':           10.0,
    'Dronagiri':      18.0,
    'Kopar Khairane':  7.5,
    'Turbhe':          9.0,
}

# ── Product catalogue (shared across stores) ────────────────────────
# Format: (product_id, product_name, category, price_INR)
PRODUCTS = [
    ('P001', 'Margherita Pizza',    'Pizza',    299, 'S001'),
    ('P002', 'Farmhouse Pizza',     'Pizza',    399, 'S001'),
    ('P003', 'Veg Extravaganza',    'Pizza',    499, 'S001'),
    ('P004', 'Peppy Paneer',        'Pizza',    449, 'S001'),
    ('P005', 'Garlic Bread',        'Sides',    149, 'S001'),
    ('P006', 'Stuffed Garlic Bread','Sides',    199, 'S001'),
    ('P007', 'Coke',               'Beverage',   60, 'S001'),
    ('P008', 'Sprite',             'Beverage',   60, 'S001'),
    ('P009', 'Brownie',            'Dessert',    99, 'S001'),
    ('P010', 'Choco Lava Cake',    'Dessert',   129, 'S001'),
    ('P011', 'Taco Mexicana',      'Sides',     169, 'S001'),
    ('P012', 'Cheese Dip',         'Sides',      49, 'S001'),
    ('P013', 'Pasta',              'Main',      249, 'S001'),
    ('P014', 'Veg Burger',         'Main',      179, 'S001'),
    ('P015', 'Chicken Burger',     'Main',      229, 'S001'),
    ('P016', 'Chicken Wings',      'Sides',     299, 'S001'),
    ('P017', 'Mousse Cake',        'Dessert',   149, 'S001'),
    ('P018', 'Iced Tea',           'Beverage',   89, 'S001'),
    ('P019', 'Water Bottle',       'Beverage',   20, 'S001'),
    ('P020', 'Paneer Wrap',        'Main',      199, 'S001'),
]

# Selection weights — higher = ordered more often
# Pizzas (P001-P004) dominate; Beverages next; Desserts least
PRODUCT_WEIGHTS = [30, 20, 15, 15, 10, 8, 20, 10, 5, 8,
                    5,  5,  8,  5,  4,  3,  2,  4,  2,  5]

print(f'Loaded {len(PRODUCTS)} products and {len(LOCATIONS)} delivery zones.')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 7, Finished, Available, Finished, False)

Loaded 20 products and 10 delivery zones.


In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 — Create Bronze Layer Folder Structure
#
# Fabric Lakehouse path convention:
#   Files/bronze/<table_name>/<table>_<store_id>_<batch_id>.json
#
# mkdir(parents=True, exist_ok=True) is idempotent — safe to re-run.
# ─────────────────────────────────────────────────────────────────────

BRONZE_ROOT = '/lakehouse/default/Files/bronze'

TABLES = ['customers', 'products', 'staff', 'orders', 'order_items', 'inventory']

def create_folders():
    for table in TABLES:
        Path(f'{BRONZE_ROOT}/{table}').mkdir(parents=True, exist_ok=True)
    print(f'Bronze folders ready under: {BRONZE_ROOT}')

create_folders()

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 8, Finished, Available, Finished, False)

Bronze folders ready under: /lakehouse/default/Files/bronze


In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 — Generate Customers Table
#
# Business rules implemented:
#  • All customers live in Navi Mumbai locality areas
#  • 'customer_since' spans last 3 years (realistic membership spread)
#  • distance_km pulled from DISTANCE_MAP (for delivery analytics)
#
# Pareto skew (20 % → 60 % revenue) is implemented in the ORDERS
# generator by choosing from a weighted customer list — so here we
# only need a flat customer pool.
# ─────────────────────────────────────────────────────────────────────

def generate_customers(n=CUSTOMER_COUNT):
    """
    Generate `n` synthetic customer records.
    Returns a pandas DataFrame with columns:
        customer_id, customer_name, email, phone,
        city, distance_km, customer_since
    """
    rows = []
    for i in range(1, n + 1):
        city = random.choice(LOCATIONS)           # random Navi Mumbai area
        rows.append([
            f'C{i:04d}',                          # e.g. C0001
            fake.name(),                           # Indian full name
            fake.email(),                          # realistic email
            fake.msisdn()[:10],                    # 10-digit mobile
            city,
            DISTANCE_MAP[city],                     # km from store
            f'{store_id}',                    
            fake.date_between(start_date='-3y', end_date='today'),
        ])

    df = pd.DataFrame(rows, columns=[
        'customer_id', 'customer_name', 'email', 'phone',
        'city', 'distance_km', 'store_id', 'customer_since',
    ])
    print(f'  customers      : {len(df):>5} rows')
    return df


customers = generate_customers()
customers.head(3)

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 9, Finished, Available, Finished, False)

  customers      :   560 rows


,customer_id,customer_name,email,phone,city,distance_km,store_id,customer_since
0,C0001,Aryan Maharaj,udantdewan@example.net,8196001338,Vashi,5.5,S001,2024-01-06
1,C0002,Thomas Sen,abeer26@example.com,2351161559,Seawoods,2.0,S001,2025-01-10
2,C0003,Farhan Memon,rehaan10@example.net,4131647525,Belapur,6.5,S001,2025-07-11


In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 — Generate Products Table
#
# Products are IDENTICAL for all stores — same product_id, name,
# category and price everywhere.  This keeps cross-store comparisons
# valid in the Gold layer without any price-harmonisation step.
# ─────────────────────────────────────────────────────────────────────

def generate_products():
    """
    Build the master product DataFrame from the PRODUCTS constant.
    Columns: product_id, product_name, category, price
    """
    df = pd.DataFrame(
        PRODUCTS,
        columns=['product_id', 'product_name', 'category', 'price', 'store_id'],
    )
    print(f'  products       : {len(df):>5} rows')
    return df


products = generate_products()
products.head(5)

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 10, Finished, Available, Finished, False)

  products       :    20 rows


,product_id,product_name,category,price,store_id
0,P001,Margherita Pizza,Pizza,299,S001
1,P002,Farmhouse Pizza,Pizza,399,S001
2,P003,Veg Extravaganza,Pizza,499,S001
3,P004,Peppy Paneer,Pizza,449,S001
4,P005,Garlic Bread,Sides,149,S001


In [7]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 — Generate Staff Table
#
# Role distribution per store:
#   1  Manager           (authorise refunds, oversee ops)
#   5  Chefs             (kitchen team)
#   2  Cashiers          (POS / billing)
#   5  Delivery Executives (majority of orders)
#   2  Cleaners          (operations support)
#
# Shifts are Morning / Evening — 'Night' is an injected DQ issue.
# ─────────────────────────────────────────────────────────────────────

def generate_staff():
    """
    Generate 15 staff records for this store.
    Columns: staff_id, staff_name, role, shift
    """
    # Build role list in the correct proportions
    roles = (
        ['Manager']            * 1 +
        ['Chef']               * 5 +
        ['Cashier']            * 2 +
        ['Delivery Executive'] * 5 +
        ['Cleaner']            * 2
    )
    rows = []
    for idx, role in enumerate(roles, start=1):
        rows.append([
            f'ST{idx:03d}',              # e.g. ST001
            f'{store_id}',
            fake.name(),
            role,
            random.choice(['Morning', 'Evening']),
        ])

    df = pd.DataFrame(rows, columns=['staff_id', 'store_id', 'staff_name', 'role', 'shift'])
    print(f'  staff          : {len(df):>5} rows')
    return df


staff = generate_staff()
staff

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 11, Finished, Available, Finished, False)

  staff          :    15 rows


,staff_id,store_id,staff_name,role,shift
0,ST001,S001,Girish Gara,Manager,Evening
1,ST002,S001,Vrishti Krish,Chef,Evening
2,ST003,S001,Lohit Rau,Chef,Morning
3,ST004,S001,Viraj Bhargava,Chef,Evening
4,ST005,S001,Zayyan Sabharwal,Chef,Morning
5,ST006,S001,Geetika Dora,Chef,Evening
6,ST007,S001,Ekanta Ray,Cashier,Evening
7,ST008,S001,Amrita Kadakia,Cashier,Evening
8,ST009,S001,Xalak Munshi,Delivery Executive,Morning
9,ST010,S001,Ekanta Shenoy,Delivery Executive,Morning


In [8]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 — Generate Orders Table
#
# Business rules implemented:
#  • Store operates 10:00 – 22:00 only
#  • Peak hours: 19:00 & 20:00 (dinner rush) — highest weights
#  • Lunch peak: 13:00 / 18:00 also elevated
#  • ~680 orders on the business date
#  • Pareto (20 % customers → 60 % orders) achieved via skewed
#    customer_weights: top-20% customers get 3× more selections
#  • Payment mix: UPI dominant (India UPI adoption), then Cash, Card
# ─────────────────────────────────────────────────────────────────────

def generate_orders(customers_df, staff_df, order_count=ORDER_COUNT):
    """
    Generate `order_count` orders for BUSINESS_DATE.
    Columns: order_id, customer_id, store_id, staff_id,
             order_timestamp, payment_mode, order_status
    """
    # ── Hour distribution weights (10 AM – 10 PM)
    hour_weights = {
        10:  5,   # Store just opened — slow
        11: 10,
        12: 20,
        13: 35,   # Lunch peak
        14: 25,
        15: 15,
        16: 20,
        17: 35,
        18: 60,   # Pre-dinner surge
        19: 90,   # ★ Peak hour
        20: 80,   # ★ Peak hour
        21: 45,   # Winding down
    }
    hours   = list(hour_weights.keys())
    weights = list(hour_weights.values())

    # ── Pareto customer skew
    # Top 20 % of customers (first 60) get weight=3, rest weight=1
    cust_ids = customers_df['customer_id'].tolist()
    pareto_cutoff = int(len(cust_ids) * 0.20)
    cust_weights  = [3] * pareto_cutoff + [1] * (len(cust_ids) - pareto_cutoff)

    staff_ids = staff_df['staff_id'].tolist()

    rows = []
    for i in range(1, order_count + 1):
        hour = random.choices(hours, weights=weights, k=1)[0]
        ts   = BUSINESS_DATE.replace(
            hour=hour,
            minute=random.randint(0, 59),
            second=random.randint(0, 59),
        )
        rows.append([
            f'O{i:05d}',
            random.choices(cust_ids, weights=cust_weights, k=1)[0],
            STORE['store_id'],
            random.choice(staff_ids),
            ts,
            random.choices(
                ['UPI', 'Cash', 'Card'],
                weights=[50, 30, 20],    # UPI most popular in India
                k=1,
            )[0],
            'Completed',
        ])

    df = pd.DataFrame(rows, columns=[
        'order_id', 'customer_id', 'store_id', 'staff_id',
        'order_timestamp', 'payment_mode', 'order_status',
    ])
    print(f'  orders         : {len(df):>5} rows')
    return df


orders = generate_orders(customers, staff)
orders.head(3)

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 12, Finished, Available, Finished, False)

  orders         :   680 rows


,order_id,customer_id,store_id,staff_id,order_timestamp,payment_mode,order_status
0,O00001,C0202,S001,ST007,2026-06-03 12:26:21.502449,UPI,Completed
1,O00002,C0380,S001,ST012,2026-06-03 20:24:57.502449,UPI,Completed
2,O00003,C0150,S001,ST006,2026-06-03 20:36:24.502449,UPI,Completed


In [9]:
# ─────────────────────────────────────────────────────────────────────
# CELL 9 — Generate Order Items Table
#
# Business rules implemented:
#  • Each order has 1–5 line items; most have 2–3 (basket-size weights)
#  • Products selected using PRODUCT_WEIGHTS (pizza-heavy)
#  • Quantity per line: 1–3 units (most orders are 1)
#  • line_amount = quantity × unit_price (no discounts at Bronze layer)
# ─────────────────────────────────────────────────────────────────────

def generate_order_items(orders_df, products_df):
    """
    Expand each order into 1-5 product line items.
    Columns: order_id, product_id, quantity, unit_price, line_amount
    """
    # Build a quick price lookup dict  {product_id: price}
    price_map = products_df.set_index('product_id')['price'].to_dict()
    product_ids = products_df['product_id'].tolist()

    rows = []
    for oid in orders_df['order_id']:
        basket_size = random.choices(
            [1, 2, 3, 4, 5],
            weights=[15, 35, 30, 15, 5],   # most orders: 2-3 items
            k=1,
        )[0]

        selected_products = random.choices(
            product_ids,
            weights=PRODUCT_WEIGHTS,
            k=basket_size,
        )

        for pid in selected_products:
            unit_price = price_map[pid]
            qty        = random.randint(1, 3)
            rows.append([
                f'{store_id}',
                oid,
                pid,
                qty,
                unit_price,
                qty * unit_price,   # line_amount
            ])

    df = pd.DataFrame(rows, columns=[
        'store_id','order_id', 'product_id', 'quantity', 'unit_price', 'line_amount',
    ])
    print(f'  order_items    : {len(df):>5} rows')
    return df


order_items = generate_order_items(orders, products)
order_items.head(5)

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 13, Finished, Available, Finished, False)

  order_items    :  1758 rows


,store_id,order_id,product_id,quantity,unit_price,line_amount
0,S001,O00001,P003,2,499,998
1,S001,O00002,P018,3,89,267
2,S001,O00002,P009,2,99,198
3,S001,O00002,P008,2,60,120
4,S001,O00003,P006,1,199,199


In [10]:
# ─────────────────────────────────────────────────────────────────────
# CELL 10 — Generate Inventory Table
#
# Inventory is derived from actual sales so it is always consistent:
#   opening_stock = random(500, 1000)   ← plausible daily stock
#   sold_qty      = sum of quantities from order_items
#   closing_stock = opening_stock − sold_qty
#
# One row per product per business date.
# ─────────────────────────────────────────────────────────────────────

def generate_inventory(order_items_df):
    """
    Build daily inventory snapshot from order_items aggregation.
    Columns: inventory_id, product_id, opening_stock,
             sold_qty, closing_stock, inventory_date
    """
    # Aggregate total sold quantity per product
    sales_agg = (
        order_items_df
        .groupby('product_id')['quantity']
        .sum()
        .reset_index()
        .rename(columns={'quantity': 'sold_qty'})
    )

    rows = []
    for _, row in sales_agg.iterrows():
        opening = random.randint(500, 1000)   # daily opening stock
        rows.append([
            f'INV_{row["product_id"]}',       # e.g. INV_P001
            row['product_id'],
            f'{store_id}',
            opening,
            int(row['sold_qty']),
            opening - int(row['sold_qty']),    # closing stock
            BUSINESS_DATE.strftime('%Y-%m-%d'),
        ])

    df = pd.DataFrame(rows, columns=[
        'inventory_id', 'product_id', 'store_id', 'opening_stock',
        'sold_qty', 'closing_stock', 'inventory_date',
    ])
    print(f'  inventory      : {len(df):>5} rows')
    return df


inventory = generate_inventory(order_items)
inventory

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 14, Finished, Available, Finished, False)

  inventory      :    20 rows


,inventory_id,product_id,store_id,opening_stock,sold_qty,closing_stock,inventory_date
0,INV_P001,P001,S001,563,632,-69,2026-06-03
1,INV_P002,P002,S001,736,320,416,2026-06-03
2,INV_P003,P003,S001,689,253,436,2026-06-03
3,INV_P004,P004,S001,979,286,693,2026-06-03
4,INV_P005,P005,S001,598,221,377,2026-06-03
5,INV_P006,P006,S001,727,157,570,2026-06-03
6,INV_P007,P007,S001,504,415,89,2026-06-03
7,INV_P008,P008,S001,613,266,347,2026-06-03
8,INV_P009,P009,S001,880,112,768,2026-06-03
9,INV_P010,P010,S001,510,156,354,2026-06-03


In [11]:
# ─────────────────────────────────────────────────────────────────────
# CELL 11 — Inject Data Quality Issues (5–10 %)
#
# This mirrors real-world Bronze data so the Silver cleansing notebook
# has something meaningful to fix.  Issue rates per spec:
#
# Customers  : 2% missing email | 2% invalid phone | 1% duplicate | 1% missing city
# Staff      : ~1% missing role | ~1% invalid shift (injected as 'Night')
# Orders     : 1% null customer_id | 1% invalid payment_mode | 1% future timestamp
#              + a few duplicate order_ids
# Order Items: 1% negative quantity | 1% zero quantity | 1% invalid product_id
# Inventory  : 1% negative closing_stock
# ─────────────────────────────────────────────────────────────────────

def inject_issues(cust_df, staff_df, orders_df, items_df, inv_df):
    """
    Mutate DataFrames in-place to simulate real-world data quality issues.
    Returns the modified DataFrames (customers is re-indexed after dup injection).
    """
    # ── Customers ──────────────────────────────────────────────────────
    # 2 % missing email
    cust_df.loc[cust_df.sample(frac=0.02, random_state=RANDOM_SEED).index, 'email'] = None
    # 2 % invalid phone (replace with text to simulate OCR / form errors)
    cust_df.loc[cust_df.sample(frac=0.02, random_state=RANDOM_SEED+1).index, 'phone'] = 'INVALID'
    # 1 % missing city
    cust_df.loc[cust_df.sample(frac=0.01, random_state=RANDOM_SEED+2).index, 'city'] = None
    # 1 % duplicate records (5 rows)
    cust_df = pd.concat([cust_df, cust_df.sample(5, random_state=RANDOM_SEED)],
                        ignore_index=True)

    # ── Staff ──────────────────────────────────────────────────────────
    # ~1 % missing role
    staff_df.loc[staff_df.sample(frac=0.07, random_state=RANDOM_SEED).index, 'role'] = None
    # ~1 % invalid shift value
    staff_df.loc[staff_df.sample(frac=0.07, random_state=RANDOM_SEED+1).index, 'shift'] = 'Night'

    # ── Orders ─────────────────────────────────────────────────────────
    # 1 % null customer_id (orphaned orders)
    orders_df.loc[orders_df.sample(frac=0.01, random_state=RANDOM_SEED).index, 'customer_id'] = None
    # 1 % invalid payment mode
    orders_df.loc[orders_df.sample(frac=0.01, random_state=RANDOM_SEED+1).index, 'payment_mode'] = 'Crypto'
    # 1 % future timestamps (system clock glitch)
    future_idx = orders_df.sample(frac=0.01, random_state=RANDOM_SEED+2).index
    orders_df.loc[future_idx, 'order_timestamp'] = (
        orders_df.loc[future_idx, 'order_timestamp'] + pd.Timedelta(days=365)
    )
    # Duplicate a handful of order_ids
    dup_orders = orders_df.sample(3, random_state=RANDOM_SEED)
    orders_df  = pd.concat([orders_df, dup_orders], ignore_index=True)

    # ── Order Items ────────────────────────────────────────────────────
    # 1 % negative quantity (data entry error)
    items_df.loc[items_df.sample(frac=0.01, random_state=RANDOM_SEED).index, 'quantity'] = -5
    # 1 % zero quantity
    items_df.loc[items_df.sample(frac=0.01, random_state=RANDOM_SEED+1).index, 'quantity'] = 0
    # 1 % invalid product_id
    items_df.loc[items_df.sample(frac=0.01, random_state=RANDOM_SEED+2).index, 'product_id'] = 'P999'

    # ── Inventory ──────────────────────────────────────────────────────
    # 1 % negative closing_stock (over-sold or counting error)
    inv_df.loc[inv_df.sample(frac=0.05, random_state=RANDOM_SEED).index, 'closing_stock'] = -100

    return cust_df, staff_df, orders_df, items_df, inv_df


customers, staff, orders, order_items, inventory = inject_issues(
    customers, staff, orders, order_items, inventory
)
print('Data quality issues injected.')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 15, Finished, Available, Finished, False)

Data quality issues injected.


In [12]:
# ─────────────────────────────────────────────────────────────────────
# CELL 12 — Add Bronze Metadata Columns
#
# Every Bronze record gets three audit columns:
#   ingestion_timestamp — wall-clock time this batch was generated
#   batch_id            — YYYYmmdd_HHMMSS string, groups all tables
#                         from a single run together
#   source_file         — exact filename this record will be written to
#
# These columns are used by the Silver notebook to:
#   • Detect and skip already-processed batches (idempotency)
#   • Track data lineage through Bronze → Silver → Gold
# ─────────────────────────────────────────────────────────────────────

def add_metadata(dfs: dict) -> str:
    """
    Add audit columns to every DataFrame in `dfs`.
    Returns the batch_id string.

    Parameters
    ----------
    dfs : dict  {table_name: DataFrame}

    Returns
    -------
    str  batch_id  e.g. '20260601_143022'
    """
    now      = datetime.now()
    # batch_id = now.strftime('%Y%m%d_%H%M%S')
    batch_id = now.strftime('%Y%m%d')

    for table_name, df in dfs.items():
        df['ingestion_timestamp'] = now.isoformat()
        df['batch_id']            = batch_id
        df['source_file']         = f'{table_name}_{store_id}_{batch_id}.json'

    return batch_id


all_dfs = {
    'customers':   customers,
    'products':    products,
    'staff':       staff,
    'orders':      orders,
    'order_items': order_items,
    'inventory':   inventory,
}

batch_id = add_metadata(all_dfs)
print(f'Batch ID: {batch_id}')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 16, Finished, Available, Finished, False)

Batch ID: 20260603


In [13]:
# ─────────────────────────────────────────────────────────────────────
# CELL 13 — Save to Bronze JSON Files
#
# File naming convention:
#   /lakehouse/default/Files/bronze/<table>/<table>_<store_id>_<batch_id>.json
#
# Example:
#   .../bronze/orders/orders_S001_20260601_143022.json
#
# orient='records'  → array of JSON objects (one per row)
# indent=2          → human-readable (easier to inspect in Fabric)
# date_format='iso' → ISO-8601 strings for all datetime columns
# ─────────────────────────────────────────────────────────────────────

def save_json(dfs: dict, batch_id: str) -> None:
    """
    Write each DataFrame to its Bronze lakehouse path as JSON.

    Parameters
    ----------
    dfs      : dict  {table_name: DataFrame}
    batch_id : str   timestamp string from add_metadata()
    """
    for table_name, df in dfs.items():
        path = f'{BRONZE_ROOT}/{table_name}/{table_name}_{store_id}_{batch_id}.json'
        df.to_json(path, orient='records', indent=2, date_format='iso')
        print(f'  Saved → {path}  ({len(df)} rows)')


save_json(all_dfs, batch_id)
print(f'\n✅ Bronze ingestion complete for {STORE["store_name"]} ({store_id}).')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 17, Finished, Available, Finished, False)

  Saved → /lakehouse/default/Files/bronze/customers/customers_S001_20260603.json  (565 rows)
  Saved → /lakehouse/default/Files/bronze/products/products_S001_20260603.json  (20 rows)
  Saved → /lakehouse/default/Files/bronze/staff/staff_S001_20260603.json  (15 rows)
  Saved → /lakehouse/default/Files/bronze/orders/orders_S001_20260603.json  (683 rows)
  Saved → /lakehouse/default/Files/bronze/order_items/order_items_S001_20260603.json  (1758 rows)
  Saved → /lakehouse/default/Files/bronze/inventory/inventory_S001_20260603.json  (20 rows)

✅ Bronze ingestion complete for Seawoods (S001).


In [14]:
# ─────────────────────────────────────────────────────────────────────
# CELL 14 — Row Count Summary & Quick Sanity Check
# ─────────────────────────────────────────────────────────────────────

summary = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in all_dfs.items()
])

print(f'\n── {STORE["store_name"]} | Batch {batch_id} ──')
print(summary.to_string(index=False))

# Quick revenue sanity
total_rev = order_items[order_items['quantity'] > 0]['line_amount'].sum()
avg_order = total_rev / len(orders)
print(f'\nApprox. total revenue : ₹{total_rev:,.0f}')
print(f'Avg order value       : ₹{avg_order:,.0f}')
print(f'Peak order hour       : {orders["order_timestamp"].dt.hour.value_counts().idxmax()}:00')

StatementMeta(, 59016fa8-e303-41b6-917b-0be26104eddc, 18, Finished, Available, Finished, False)


── Seawoods | Batch 20260603 ──
      table  rows  columns
  customers   565       11
   products    20        8
      staff    15        8
     orders   683       10
order_items  1758        9
  inventory    20       10

Approx. total revenue : ₹827,343
Avg order value       : ₹1,211
Peak order hour       : 19:00
